# Neural Speech Decoding — Feature Extraction Pipeline
**Dataset:** SingleWordProductionDutch-iBIDS  
**Pipeline:** sEEG → High-Gamma Features → Mel Spectrogram → Aligned Feature Matrices  
**Authors:** Adapted for LOSO cross-validation pipeline


In [ ]:
import os
import pandas as pd
import numpy as np
import numpy.matlib as matlib
import scipy
import scipy.signal
import scipy.stats
import scipy.io.wavfile
import scipy.fftpack

from pynwb import NWBHDF5IO
import MelFilterBank as mel

# Fast Hilbert transform — extends signal length to next power of 2 for speed
hilbert3 = lambda x: scipy.signal.hilbert(
    x, scipy.fftpack.next_fast_len(len(x)), axis=0
)[:len(x)]

print("✅ Imports OK")


## ⚙️ Configuration
Set all paths and hyperparameters here before running the pipeline.

In [ ]:
# ── Windowing parameters ──────────────────────────────────────────────────────
winL        = 0.05   # Window length (seconds)
frameshift  = 0.01   # Frame shift (seconds)
modelOrder  = 4      # Temporal context (frames before/after current window)
stepSize    = 5      # Step size for temporal context stacking

# ── Paths ─────────────────────────────────────────────────────────────────────
path_bids   = r'C:/Users/PC/Desktop/spain/dataset/test/SingleWordProductionDutch-iBIDS/'
path_output = r'C:/Users/PC/Desktop/spain/dataset/test/features'

participants_tsv = os.path.join(path_bids, 'participants.tsv')

print(f"BIDS root : {path_bids}")
print(f"Output dir: {path_output}")


## 🔧 Helper Functions

In [ ]:
def extractHG(data, sr, windowLength=0.05, frameshift=0.01):
    """
    Window data and extract High-Gamma (70–170 Hz) envelope via Hilbert transform.

    Parameters
    ----------
    data : array (samples, channels)
    sr   : int — sampling rate
    windowLength : float — window length in seconds
    frameshift   : float — frame shift in seconds

    Returns
    -------
    feat : array (windows, channels)
    """
    # Linear detrend
    data = scipy.signal.detrend(data, axis=0)

    numWindows = int(np.floor((data.shape[0] - windowLength * sr) / (frameshift * sr)))

    # Bandpass: High-Gamma (70–170 Hz)
    sos = scipy.signal.iirfilter(4, [70/(sr/2), 170/(sr/2)], btype='bandpass', output='sos')
    data = scipy.signal.sosfiltfilt(sos, data, axis=0)

    # Notch: 1st harmonic of line noise (98–102 Hz)
    sos = scipy.signal.iirfilter(4, [98/(sr/2), 102/(sr/2)], btype='bandstop', output='sos')
    data = scipy.signal.sosfiltfilt(sos, data, axis=0)

    # Notch: 2nd harmonic of line noise (148–152 Hz)
    sos = scipy.signal.iirfilter(4, [148/(sr/2), 152/(sr/2)], btype='bandstop', output='sos')
    data = scipy.signal.sosfiltfilt(sos, data, axis=0)

    # Analytic envelope
    data = np.abs(hilbert3(data))

    feat = np.zeros((numWindows, data.shape[1]))
    for win in range(numWindows):
        start = int(np.floor((win * frameshift) * sr))
        stop  = int(np.floor(start + windowLength * sr))
        feat[win, :] = np.mean(data[start:stop, :], axis=0)

    return feat

print("✅ extractHG defined")


In [ ]:
def stackFeatures(features, modelOrder=4, stepSize=5):
    """
    Add temporal context by stacking neighbouring feature vectors.

    Parameters
    ----------
    features   : array (windows, channels)
    modelOrder : int — context frames before/after current window
    stepSize   : int — stride for context sampling

    Returns
    -------
    featStacked : array (windows, channels * (2*modelOrder+1))
    """
    n_out = features.shape[0] - (2 * modelOrder * stepSize)
    n_feat = (2 * modelOrder + 1) * features.shape[1]
    featStacked = np.zeros((n_out, n_feat))

    for fNum, i in enumerate(range(modelOrder * stepSize,
                                   features.shape[0] - modelOrder * stepSize)):
        ef = features[i - modelOrder*stepSize : i + modelOrder*stepSize + 1 : stepSize, :]
        featStacked[fNum, :] = ef.flatten()

    return featStacked

print("✅ stackFeatures defined")


In [ ]:
def downsampleLabels(labels, sr, windowLength=0.05, frameshift=0.01):
    """
    Downsample string labels using the mode within each window.

    Parameters
    ----------
    labels       : array of str — sample-level labels
    sr           : int — sampling rate
    windowLength : float
    frameshift   : float

    Returns
    -------
    newLabels : array of str — window-level labels
    """
    numWindows = int(np.floor((labels.shape[0] - windowLength * sr) / (frameshift * sr)))
    newLabels  = np.empty(numWindows, dtype="S15")

    for w in range(numWindows):
        start = int(np.floor((w * frameshift) * sr))
        stop  = int(np.floor(start + windowLength * sr))
        newLabels[w] = (
            scipy.stats.mode(labels[start:stop])[0][0]
            .encode("ascii", errors="ignore")
            .decode()
        )
    return newLabels

print("✅ downsampleLabels defined")


In [ ]:
def extractMelSpecs(audio, sr, windowLength=0.05, frameshift=0.01, numFilter=23):
    """
    Extract log-mel spectrogram from audio.

    Parameters
    ----------
    audio        : array — audio time series
    sr           : int — audio sampling rate
    windowLength : float
    frameshift   : float
    numFilter    : int — mel filter bank size

    Returns
    -------
    spectrogram : array (numWindows, numFilter)
    """
    numWindows = int(np.floor((audio.shape[0] - windowLength * sr) / (frameshift * sr)))
    win = scipy.hanning(int(np.floor(windowLength * sr + 1)))[:-1]
    spectrogram = np.zeros(
        (numWindows, int(np.floor(windowLength * sr / 2 + 1))), dtype='complex'
    )

    for w in range(numWindows):
        start_audio = int(np.floor((w * frameshift) * sr))
        stop_audio  = int(np.floor(start_audio + windowLength * sr))
        a = audio[start_audio:stop_audio]
        spectrogram[w, :] = np.fft.rfft(win * a)

    mfb = mel.MelFilterBank(spectrogram.shape[1], numFilter, sr)
    spectrogram = np.abs(spectrogram)
    spectrogram = (mfb.toLogMels(spectrogram)).astype('float')
    return spectrogram

print("✅ extractMelSpecs defined")


In [ ]:
def nameVector(elecs, modelOrder=4):
    """
    Build electrode name list with temporal context suffixes (T-N … T0 … T+N).

    Parameters
    ----------
    elecs      : array of str — electrode names, shape (n_elec, 1)
    modelOrder : int

    Returns
    -------
    names : array of str, shape (n_elec * (2*modelOrder+1),)
    """
    names = matlib.repmat(elecs.astype(np.dtype(('U', 10))), 1, 2 * modelOrder + 1).T
    for i, off in enumerate(range(-modelOrder, modelOrder + 1)):
        names[i, :] = [e[0] + 'T' + str(off) for e in elecs]
    return names.flatten()

print("✅ nameVector defined")


---
## 🚀 Main Pipeline

Iterates over all participants, loads NWB data, extracts High-Gamma features and
mel spectrograms, aligns them in time, and saves `.npy` outputs per subject.


In [ ]:
participants = pd.read_csv(participants_tsv, delimiter='\t')
print(f"Found {len(participants)} participants:")
print(participants['participant_id'].values)


In [ ]:
for p_id, participant in enumerate(participants['participant_id']):
    print(f"\n{'='*60}")
    print(f"Processing: {participant}  ({p_id+1}/{len(participants)})")
    print('='*60)

    # ── 1. Load NWB file ──────────────────────────────────────────────────────
    nwb_path = os.path.join(
        path_bids, participant, 'ieeg',
        f'{participant}_task-wordProduction_ieeg.nwb'
    )
    io = NWBHDF5IO(nwb_path, 'r')
    nwbfile = io.read()

    eeg    = nwbfile.acquisition['iEEG'].data[:]
    eeg_sr = 1024

    audio    = nwbfile.acquisition['Audio'].data[:]
    audio_sr = 48000

    words = np.array(nwbfile.acquisition['Stimulus'].data[:], dtype=str)
    io.close()

    print(f"  sEEG  : {eeg.shape}  @ {eeg_sr} Hz")
    print(f"  Audio : {audio.shape} @ {audio_sr} Hz")

    # ── 2. Load channel names ─────────────────────────────────────────────────
    ch_path = os.path.join(
        path_bids, participant, 'ieeg',
        f'{participant}_task-wordProduction_channels.tsv'
    )
    channels = np.array(pd.read_csv(ch_path, delimiter='\t')['name'])
    print(f"  Channels: {len(channels)}")

    # ── 3. Extract High-Gamma features ────────────────────────────────────────
    print("  → Extracting High-Gamma features...")
    feat = extractHG(eeg, eeg_sr, windowLength=winL, frameshift=frameshift)
    print(f"     HG feat shape: {feat.shape}")

    # ── 4. Stack temporal context ─────────────────────────────────────────────
    feat = stackFeatures(feat, modelOrder=modelOrder, stepSize=stepSize)
    print(f"     Stacked feat shape: {feat.shape}")

    # ── 5. Resample & scale audio ─────────────────────────────────────────────
    target_SR = 16000
    audio     = scipy.signal.decimate(audio, int(audio_sr / target_SR))
    audio_sr  = target_SR
    scaled    = np.int16(audio / np.max(np.abs(audio)) * 32767)
    print(f"  → Audio resampled to {audio_sr} Hz, shape: {scaled.shape}")

    # ── 6. Extract mel spectrogram ────────────────────────────────────────────
    print("  → Extracting mel spectrogram...")
    melSpec = extractMelSpecs(scaled, audio_sr, windowLength=winL, frameshift=frameshift)
    print(f"     Mel spec shape (before alignment): {melSpec.shape}")

    # ── 7. Align mel spec to EEG features ────────────────────────────────────
    melSpec = melSpec[modelOrder*stepSize : melSpec.shape[0] - modelOrder*stepSize, :]

    if melSpec.shape[0] != feat.shape[0]:
        tLen    = np.min([melSpec.shape[0], feat.shape[0]])
        melSpec = melSpec[:tLen, :]
        feat    = feat[:tLen, :]
        print(f"     ⚠ Length mismatch — trimmed to {tLen} frames")

    print(f"     Final aligned shapes → feat: {feat.shape}, mel: {melSpec.shape}")

    # ── 8. Feature names ──────────────────────────────────────────────────────
    feature_names = nameVector(channels[:, None], modelOrder=modelOrder)

    # ── 9. Save outputs ───────────────────────────────────────────────────────
    out_dir = os.path.join(path_output, participant)
    os.makedirs(out_dir, exist_ok=True)

    np.save(os.path.join(out_dir, f'{participant}_feat.npy'),       feat)
    np.save(os.path.join(out_dir, f'{participant}_words.npy'),      words)
    np.save(os.path.join(out_dir, f'{participant}_melSpec.npy'),    melSpec)
    np.save(os.path.join(out_dir, f'{participant}_feat_names.npy'), feature_names)

    print(f"  ✅ Saved to: {out_dir}")

print("\n🎉 All participants processed.")


## 🔍 Sanity Check — Inspect Last Subject

In [ ]:
# Reload the last saved subject and print shapes
last_id  = participants['participant_id'].iloc[-1]
out_dir  = os.path.join(path_output, last_id)

feat_check  = np.load(os.path.join(out_dir, f'{last_id}_feat.npy'))
mel_check   = np.load(os.path.join(out_dir, f'{last_id}_melSpec.npy'))
words_check = np.load(os.path.join(out_dir, f'{last_id}_words.npy'))
names_check = np.load(os.path.join(out_dir, f'{last_id}_feat_names.npy'))

print(f"Subject       : {last_id}")
print(f"feat shape    : {feat_check.shape}")
print(f"melSpec shape : {mel_check.shape}")
print(f"words shape   : {words_check.shape}")
print(f"feat_names    : {names_check.shape}  (first 5: {names_check[:5]})")
print(f"\nfeat  — min={feat_check.min():.4f}, max={feat_check.max():.4f}, mean={feat_check.mean():.4f}")
print(f"melSpec — min={mel_check.min():.4f},  max={mel_check.max():.4f},  mean={mel_check.mean():.4f}")
